In this work, we use transformer model to integrate gene expression and TCR amino acid sequences

Getting gene data

In [1]:
# %matplotlib inline

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import os 
import numpy as np

import pandas as pd
# import seaborn as sb
import matplotlib.pyplot as pl

import scanpy as sc

import anndata as ad

from scipy.sparse import csr_matrix
from matplotlib import rcParams
from matplotlib import colors

sc.settings.verbosity = 3


In [2]:
gene_TCR = ad.read_h5ad('../10Xdatasets/gex_merge_log1_5000_genes_all_peptides.h5ad')
gene_TCR

/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/anndata/_core/anndata.py:1754: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 145479 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', 'A0201_I

In [3]:
gene_TCR = gene_TCR[gene_TCR.obs.donor == 'donor4']
gene_TCR

View of AnnData object with n_obs × n_vars = 21393 × 5000
    obs: 'v_gene_TRA', 'v_gene_TRB', 'd_gene_TRA', 'd_gene_TRB', 'j_gene_TRA', 'j_gene_TRB', 'c_gene_TRA', 'c_gene_TRB', 'cdr3_TRA', 'cdr3_TRB', 'cdr3_nt_TRA', 'cdr3_nt_TRB', 'umis_TRA', 'umis_TRB', 'donor', 'cell_clono_cdr3_aa', 'cell_clono_cdr3_nt', 'CD3', 'CD19', 'CD45RA', 'CD4', 'CD8a', 'CD14', 'CD45RO', 'CD279_PD-1', 'IgG1', 'IgG2a', 'IgG2b', 'CD127', 'CD197_CCR7', 'HLA-DR', 'A0101_VTEHDTLLY_IE-1_CMV', 'A0201_KTWGQYWQV_gp100_Cancer', 'A0201_ELAGIGILTV_MART-1_Cancer', 'A0201_CLLWSFQTSA_Tyrosinase_Cancer', 'A0201_IMDQVPFSV_gp100_Cancer', 'A0201_SLLMWITQV_NY-ESO-1_Cancer', 'A0201_KVAELVHFL_MAGE-A3_Cancer', 'A0201_KVLEYVIKV_MAGE-A1_Cancer', 'A0201_CLLGTYTQDV_Kanamycin-B-dioxygenase', 'A0201_LLDFVRFMGV_EBNA-3B_EBV', 'A0201_LLMGTLGIVC_HPV-16E7_82-91', 'A0201_CLGGLLTMV_LMP-2A_EBV', 'A0201_YLLEMLWRL_LMP1_EBV', 'A0201_FLYALALLL_LMP2A_EBV', 'A0201_GILGFVFTL_Flu-MP_Influenza', 'A0201_GLCTLVAML_BMLF1_EBV', 'A0201_NLVPMVATV_pp65_CMV', '

In [4]:
gene_TCR.obs.donor

barcode
TTGGAACTCCGTCATC-8    donor4
GACGTTAAGATGTCGG-8    donor4
CGTGTAAAGGGCACTA-6    donor4
ATCCGAAGTTTGGGCC-2    donor4
CTCGAAAAGGAGTAGA-3    donor4
                       ...  
GAAGCAGAGCAGGCTA-3    donor4
CAGTCCTTCATCACCC-8    donor4
GACTACACACGGTAAG-3    donor4
ATCGAGTAGTTTCCTT-4    donor4
ACCCACTTCTTGGGTA-3    donor4
Name: donor, Length: 21393, dtype: category
Categories (1, object): ['donor4']

In [5]:
gene = pd.DataFrame(gene_TCR.X.todense())
gene

,0,1,2,3,4,5,6,7,8,9,...,4990,4991,4992,4993,4994,4995,4996,4997,4998,4999
0,0.0,0.693147,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.693147,0.000000,0.0,0.0
1,0.0,0.000000,0.0,0.000000,0.0,0.0,0.693147,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,1.791759,0.693147,0.0,0.0
2,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,1.609438,0.693147,0.0,0.0
3,0.0,0.693147,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.693147,0.0,...,0.693147,0.0,0.000000,0.0,0.000000,0.0,0.693147,1.098612,0.0,0.0
4,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.693147,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,1.609438,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21388,0.0,0.693147,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0
21389,0.0,0.693147,0.0,0.693147,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.693147,0.0,1.791759,1.386294,0.0,0.0
21390,0.0,0.693147,0.0,0.000000,0.0,0.0,0.000000,0.693147,0.693147,0.0,...,0.000000,0.0,0.693147,0.0,0.000000,0.0,0.693147,0.693147,0.0,0.0
21391,0.0,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.0,1.098612,0.693147,0.0,0.0


In [6]:
tcr_seq = gene_TCR.obs[['cdr3_TRB']]
tcr_seq

,cdr3_TRB
barcode,
TTGGAACTCCGTCATC-8,CASSTGAGEQYF
GACGTTAAGATGTCGG-8,CASSLRGANNEQFF
CGTGTAAAGGGCACTA-6,CAWRGFRGYTF
ATCCGAAGTTTGGGCC-2,CATSDRLAGGELFF
CTCGAAAAGGAGTAGA-3,CASSLWTGPQETQYF
...,...
GAAGCAGAGCAGGCTA-3,CATSDRLAGGELFF
CAGTCCTTCATCACCC-8,CASSYLAGDFTDTQYF
GACTACACACGGTAAG-3,CASRTGLASTDTQYF


In [7]:
import tensorflow as tf

2026-02-26 12:26:45.061367: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 12:26:45.214961: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-02-26 12:26:45.257815: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Reshape, Conv2D, Conv2DTranspose, Flatten, Dropout
from tensorflow.keras.initializers import HeNormal

# Define input layer
input_gex = Input(shape=(100,))
gex = Dense(units=64, activation='relu', kernel_initializer=HeNormal())(input_gex)
gex = Reshape(target_shape=(8,8,1))(gex)

# Convolutional layers
gex = Conv2D(filters=64, kernel_size=3, strides=1, activation='relu', padding="same")(gex)
gex = Conv2D(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(gex)

gex = Flatten()(gex)
hidden_layer = Dense(units=225, activation='relu', kernel_initializer=HeNormal())(gex)

# Transposed Convolutional layers
tcr = Reshape(target_shape=(15,15,1))(hidden_layer)
tcr = Conv2DTranspose(filters=16, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Conv2DTranspose(filters=32, kernel_size=3, strides=1, activation='relu', padding="same")(tcr)
tcr = Dropout(rate=0.2)(tcr)
tcr = Flatten()(tcr)
tcr = Dense(units=150, activation='relu', kernel_initializer=HeNormal())(tcr)
tcr = Dense(units=130, activation='relu')(tcr)  # Ensure proper activation
tcr = Dense(units=121)(tcr)
# Define model
model = Model(inputs=input_gex, outputs=tcr)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0015),
              loss='mse')

# Check layer names
model.summary()


Model: "model"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 input_1 (InputLayer)        [(None, 100)]             0         
                                                                 
 dense (Dense)               (None, 64)                6464      
                                                                 
 reshape (Reshape)           (None, 8, 8, 1)           0         
                                                                 
 conv2d (Conv2D)             (None, 8, 8, 64)          640       
                                                                 
 conv2d_1 (Conv2D)           (None, 8, 8, 32)          18464     
                                                                 
 flatten (Flatten)           (None, 2048)              0         
                                                                 
 dense_1 (Dense)             (None, 225)               461025

2026-02-26 12:26:47.009064: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F AVX512_VNNI FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-26 12:26:47.164987: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1616] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 18333 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3090, pci bus id: 0000:17:00.0, compute capability: 8.6


In [9]:
AE_tcr = pd.read_csv("../AE_emb_TRB_all_peptides_10X_donor_4_only.csv")
AE_tcr

,0,1,2,3,4,5,6,7,8,9,...,111,112,113,114,115,116,117,118,119,120
0,-578.545800,643.72100,19.97399,36.18518,522.106450,308.46360,-121.881230,-504.18234,-44.327106,-141.961990,...,611.465760,558.75890,-629.210700,-135.228560,-136.85368,-77.605034,-663.28406,194.074050,-332.754880,129.144600
1,-260.955440,-166.87968,-397.96503,-512.32580,-26.874681,-199.71866,50.246600,-209.96062,37.158900,16.608343,...,-269.681760,833.96840,543.433100,313.055000,-414.39044,-168.984710,743.24600,1380.880200,-28.762701,-353.440860
2,-392.291780,221.41151,230.58847,-48.90060,-467.185180,-175.72255,-417.281340,362.83102,-486.289800,-66.351900,...,14.493301,-136.76251,-15.745383,-122.987620,457.00250,-410.685500,358.23334,1665.129500,-788.868500,-644.535000
3,-512.594400,-576.57745,-841.85034,-472.11697,-47.690357,-245.14503,299.702880,-409.62897,-274.262540,191.513120,...,-605.121300,1250.64930,-388.323060,282.790900,-704.64404,37.460087,647.88403,1549.309700,352.393370,-108.645460
4,129.581150,-313.10944,268.98666,766.28190,-473.843140,979.15393,390.149500,479.94060,263.797360,-736.299740,...,-474.085820,748.01800,809.877260,-49.473446,631.27747,-380.982940,274.19620,-88.337395,-572.748600,-13.084727
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21388,-512.594400,-576.57745,-841.85034,-472.11697,-47.690357,-245.14503,299.702880,-409.62897,-274.262540,191.513120,...,-605.121300,1250.64930,-388.323060,282.790900,-704.64404,37.460087,647.88403,1549.309700,352.393370,-108.645460
21389,226.274460,956.16350,176.05058,-689.92930,162.394580,608.82800,-269.477200,-744.45310,-219.842960,835.156430,...,31.987175,508.31200,-341.937840,-267.425540,-786.67950,-816.931150,291.38525,466.537630,-74.314720,-342.459840
21390,1080.433200,373.86273,-65.57250,-259.77710,35.480900,-554.94800,-441.794220,93.13524,-163.760600,155.478740,...,-251.648390,687.42700,-52.242027,-690.953370,523.99010,-314.377530,-316.33014,697.732060,40.809720,-1766.797200
21391,55.882042,-766.63635,229.29404,-177.78714,634.726900,-295.03427,92.827690,443.62744,244.667280,-787.343700,...,385.697270,895.55615,-700.939100,-126.451060,-803.60010,133.600590,-130.16379,348.644780,-400.828280,38.264300


In [10]:
import numpy as np
from sklearn.decomposition import NMF

# Generate random non-negative data
data = gene.to_numpy()

# Initialize the NMF model
n_components = 100
model_nmf = NMF(n_components=n_components, init='random', random_state=0)

# Fit the model to the data
W = model_nmf.fit_transform(data)
H = model_nmf.components_

# Display the results
print("Basis matrix (W):\n", W)
print("Coefficients matrix (H):\n", H)


/home/phile/miniconda3/envs/integration/lib/python3.9/site-packages/sklearn/decomposition/_nmf.py:1742: ConvergenceWarning: Maximum number of iterations 200 reached. Increase it to improve convergence.
  warnings.warn(


Basis matrix (W):
 [[0.65588474 0.5470397  0.0521722  ... 0.07750414 0.         0.        ]
 [0.71132225 0.5102544  0.         ... 0.00402349 0.00669418 0.00819835]
 [0.7988329  0.69548815 0.28390643 ... 0.22728091 0.02105804 0.01490938]
 ...
 [0.90891445 0.08494315 0.67154783 ... 0.20570257 0.00487804 0.00460975]
 [0.         0.01184397 0.         ... 0.         0.00553914 0.        ]
 [0.6582172  0.22561976 0.95175385 ... 0.12736581 0.01004636 0.00202849]]
Coefficients matrix (H):
 [[0.         0.05022614 0.         ... 0.         0.         0.        ]
 [0.         0.00116642 0.         ... 0.         0.         0.        ]
 [0.         0.04755427 0.         ... 0.         0.         0.        ]
 ...
 [0.         0.         0.         ... 0.         0.         0.00388899]
 [0.00337751 0.         0.         ... 0.         0.         0.00996204]
 [0.         0.         0.         ... 0.         0.         0.04406664]]


In [11]:
W = pd.DataFrame(W)
W

,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,0.655885,0.547040,0.052172,0.302054,0.000000,0.000000,0.007344,0.008392,0.000000,0.125792,...,0.000000,0.000000,0.001850,0.009266,0.000000,0.000000,0.006932,0.077504,0.000000,0.000000
1,0.711322,0.510254,0.000000,0.254477,0.012534,0.001916,0.000647,0.000000,0.000000,0.178682,...,0.000000,0.000000,0.064455,0.010931,0.047753,0.001505,0.008079,0.004023,0.006694,0.008198
2,0.798833,0.695488,0.283906,0.526086,0.000000,0.001700,0.006966,0.000000,0.000000,0.161375,...,0.054184,0.000000,0.106702,0.000000,0.047100,0.027175,0.013801,0.227281,0.021058,0.014909
3,0.127750,0.209531,0.764994,0.085311,0.557679,0.000000,0.011223,0.011077,0.219407,0.078343,...,0.033616,0.027941,0.069957,0.028916,0.000000,0.008010,0.011309,0.078524,0.000000,0.000340
4,0.872419,0.386753,0.000000,0.542574,0.000000,0.018086,0.011532,0.006256,0.000000,0.152062,...,0.000000,0.000000,0.074202,0.007931,0.020739,0.013283,0.016447,0.076225,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21388,0.000000,0.000000,0.000000,0.000000,0.000000,0.000169,0.000171,0.029582,0.000000,0.000000,...,0.076934,0.079335,0.000333,0.027689,0.001277,0.043940,0.000069,0.003663,0.000000,0.000000
21389,0.751038,0.191501,0.000000,0.258498,0.467071,0.000000,0.012280,0.010047,0.118869,0.183304,...,0.034551,0.000000,0.000000,0.045172,0.000000,0.000000,0.000375,0.184599,0.009241,0.003461
21390,0.908914,0.084943,0.671548,0.000415,0.256613,0.001464,0.000766,0.017459,0.113861,0.169165,...,0.035153,0.052311,0.000000,0.045544,0.000000,0.029628,0.006845,0.205703,0.004878,0.004610
21391,0.000000,0.011844,0.000000,0.000000,0.100574,0.007823,0.011897,0.004592,0.026068,0.125654,...,0.033723,0.000000,0.001601,0.000000,0.033015,0.027142,0.000000,0.000000,0.005539,0.000000


In [12]:

# es_callback = EarlyStopping(monitor= 'val_auc', patience=20, restore_best_weights=True)
reduce_learning_rate = tf.keras.callbacks.ReduceLROnPlateau(factor=0.1, patience=50, monitor='loss', min_delta=100)

history = model.fit(W,AE_tcr, 
                epochs=8000, 
                batch_size=256, 
                shuffle = True,
                # callbacks=[es_callback, checkpoint,reduce_learning_rate])
                # callbacks=[reduce_learning_rate]
                )

Epoch 1/8000


2026-02-26 12:29:04.898831: I tensorflow/stream_executor/cuda/cuda_blas.cc:1614] TensorFloat-32 will be used for the matrix multiplication. This will only be logged once.
2026-02-26 12:29:05.607442: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8100
2026-02-26 12:29:06.574117: I tensorflow/core/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


84/84 [==============================] - 3s 5ms/step - loss: 256805.4219
Epoch 2/8000
84/84 [==============================] - 0s 4ms/step - loss: 222412.6719
Epoch 3/8000
84/84 [==============================] - 0s 4ms/step - loss: 221953.0000
Epoch 4/8000
84/84 [==============================] - 0s 4ms/step - loss: 221866.1406
Epoch 5/8000
84/84 [==============================] - 0s 4ms/step - loss: 221806.9375
Epoch 6/8000
84/84 [==============================] - 0s 4ms/step - loss: 221573.5156
Epoch 7/8000
84/84 [==============================] - 0s 4ms/step - loss: 203273.8438
Epoch 8/8000
84/84 [==============================] - 0s 4ms/step - loss: 187258.6406
Epoch 9/8000
84/84 [==============================] - 0s 4ms/step - loss: 186719.3438
Epoch 10/8000
84/84 [==============================] - 0s 4ms/step - loss: 186189.1406
Epoch 11/8000
84/84 [==============================] - 0s 4ms/step - loss: 181602.5625
Epoch 12/8000
84/84 [==============================] - 0s 4ms/ste

In [13]:
for layer in model.layers:
    print(layer.name)

input_1
dense
reshape
conv2d
conv2d_1
flatten
dense_1
reshape_1
conv2d_transpose
conv2d_transpose_1
dropout
flatten_1
dense_2
dense_3
dense_4


In [14]:
from tensorflow.keras.models import Model
latent_model = Model(inputs=input_gex, outputs=hidden_layer)


In [15]:
model.predict( W.iloc[1:2])

1/1 [==============================] - 0s 185ms/step


array([[-2.24208359e+02, -1.97359116e+02, -2.10799942e+02,
        -3.08458618e+02, -1.65448837e+01, -2.63240845e+02,
        -1.70755768e+02, -1.69316559e+02, -2.39512787e+02,
        -2.39700348e+02, -3.12680016e+01,  4.28726654e+02,
        -2.49372681e+02,  1.47123840e+02, -3.47111816e+02,
        -2.10083141e+01, -9.39030228e+01, -1.13395882e+02,
        -1.28423309e+02,  3.45298431e+02, -9.17555923e+01,
        -7.97398682e+01, -7.08769043e+02, -3.73078705e+02,
        -2.63861511e+02, -9.31281921e+02,  9.07449417e+01,
        -1.29708176e+02, -2.50253189e+02,  4.84682693e+01,
        -7.77644226e+02,  5.12047913e+02, -5.03804932e+01,
        -1.02488070e+01,  8.89728638e+02,  1.85931969e+01,
         1.86489243e+02,  1.68347893e+01, -2.44829178e+01,
        -1.18158981e+02,  5.75618286e+02, -2.63524292e+02,
         1.15523758e+02,  7.16984253e+01,  1.29332748e+02,
        -1.12879974e+02,  8.45600815e+01, -1.97439041e+02,
         1.70638428e+02,  6.25315063e+02, -3.16162018e+0

In [16]:
model.predict( W.iloc[4:5])

1/1 [==============================] - 0s 23ms/step


array([[ 2.99025097e+01, -1.29823334e+02,  5.04053375e+02,
         1.02576099e+03, -4.59208252e+02,  9.47378784e+02,
         5.51705261e+02,  6.47405273e+02,  4.24101044e+02,
        -7.85735535e+02,  5.21903076e+02, -1.05286133e+02,
         6.43223755e+02,  6.10786255e+02, -2.51719131e+02,
         3.03378315e+01,  2.37277206e+02,  3.74429993e+02,
        -3.30582916e+02,  5.72548096e+02, -7.45910767e+02,
         5.96614380e+02,  6.01103149e+02,  5.87497314e+02,
        -4.40165161e+02, -6.34387939e+02,  9.74856140e+02,
        -7.61237244e+02, -7.79006165e+02, -1.26725305e+03,
        -9.25203629e+01, -1.19824005e+02,  4.54723663e+02,
         2.67174408e+02, -5.93422119e+02, -2.20216827e+02,
        -1.88264389e+02,  3.54315979e+02, -6.14906738e+02,
        -1.84951324e+01, -1.29966751e+02, -1.19321472e+02,
         2.81959564e+02, -4.46191528e+02,  1.52803772e+02,
        -3.38887299e+02,  1.73328278e+02,  5.09690002e+02,
         7.32550354e+01,  2.32565048e+02, -1.55338165e+0

In [17]:
integration_pred = latent_model.predict( W)

669/669 [==============================] - 1s 819us/step


In [18]:
pd.DataFrame(integration_pred[1:50,1:50])

,0,1,2,3,4,5,6,7,8,9,...,39,40,41,42,43,44,45,46,47,48
0,0.000000,122.068420,144.880676,0.0,0.0,0.0,235.171188,133.973587,0.0,54.973404,...,0.0,110.244179,0.0,0.0,0.000000,2.161657,0.0,100.892456,0.0,0.000000
1,0.000000,177.388382,175.409531,0.0,0.0,0.0,264.593536,50.165634,0.0,37.211384,...,0.0,97.916641,0.0,0.0,158.319702,1.101223,0.0,0.000000,0.0,0.000000
2,0.000000,79.017914,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,0.000000,...,0.0,0.000000,0.0,0.0,30.997129,0.000000,0.0,0.000000,0.0,0.000000
3,0.000000,144.278122,73.185326,0.0,0.0,0.0,172.237823,0.000000,0.0,0.000000,...,0.0,24.806492,0.0,0.0,178.823868,151.895065,0.0,8.056357,0.0,0.000000
4,0.000000,94.950539,68.161316,0.0,0.0,0.0,183.732925,89.298347,0.0,85.119553,...,0.0,159.515182,0.0,0.0,111.759735,29.540537,0.0,83.102173,0.0,0.000000
5,0.000000,108.689705,104.009605,0.0,0.0,0.0,177.110977,123.488007,0.0,128.793594,...,0.0,88.715446,0.0,0.0,96.653870,0.000000,0.0,126.614189,0.0,0.000000
6,0.000000,99.639862,166.959854,0.0,0.0,0.0,35.398140,73.442856,0.0,0.000000,...,0.0,0.000000,0.0,0.0,68.998665,7.713584,0.0,0.000000,0.0,0.000000
7,0.000000,33.599720,24.182539,0.0,0.0,0.0,254.607193,116.135643,0.0,0.000000,...,0.0,44.849846,0.0,0.0,0.000000,7.122348,0.0,27.922842,0.0,0.000000
8,322.369934,308.229858,289.871307,0.0,0.0,0.0,260.251099,0.000000,0.0,173.091080,...,0.0,262.992004,0.0,0.0,262.000336,166.428574,0.0,0.000000,0.0,392.639282
9,0.000000,97.831253,118.094978,0.0,0.0,0.0,198.399567,153.333588,0.0,47.584805,...,0.0,271.500000,0.0,0.0,53.936348,34.616501,0.0,156.498688,0.0,0.000000


In [19]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])

In [20]:
integration_pred.shape

(21393, 225)

In [21]:
integration_pred = integration_pred.reshape([integration_pred.shape[0],-1])
integration_pred.shape

(21393, 225)

In [22]:
pd.DataFrame(integration_pred)

,0,1,2,3,4,5,6,7,8,9,...,215,216,217,218,219,220,221,222,223,224
0,0.000000,0.0000,44.936493,91.680641,0.0,0.0,0.0,53.601440,38.394032,0.0,...,0.0,0.000000,0.0,27.987392,122.077484,0.0,0.0,0.000000,0.000000,0.000000
1,0.000000,0.0000,122.068420,144.880676,0.0,0.0,0.0,235.171188,133.973587,0.0,...,0.0,0.000000,0.0,0.000000,58.173519,0.0,0.0,0.000000,0.000000,92.982040
2,0.000000,0.0000,177.388382,175.409531,0.0,0.0,0.0,264.593536,50.165634,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,0.000000,50.614620
3,0.000000,0.0000,79.017914,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,55.561527,0.0,97.908569,83.885185,0.0,0.0,35.345871,0.000000,0.000000
4,0.000000,0.0000,144.278122,73.185326,0.0,0.0,0.0,172.237823,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,97.810318,0.0,0.0,0.000000,0.000000,18.829268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21388,0.000000,0.0000,83.636475,0.000000,0.0,0.0,0.0,0.000000,0.000000,0.0,...,0.0,108.500023,0.0,115.580772,166.173477,0.0,0.0,71.598198,0.000000,0.000000
21389,15.489287,0.0000,82.227959,85.610451,0.0,0.0,0.0,161.260818,0.000000,0.0,...,0.0,0.000000,0.0,0.000000,218.587601,0.0,0.0,0.000000,0.000000,62.542667
21390,0.000000,208.6073,189.220612,269.774323,0.0,0.0,0.0,19.516222,100.878342,0.0,...,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.000000,55.067757,68.850571
21391,0.000000,0.0000,101.666344,159.804276,0.0,0.0,0.0,189.779175,125.373993,0.0,...,0.0,0.000000,0.0,0.000000,56.488537,0.0,0.0,0.000000,0.000000,0.000000


In [23]:
pd.DataFrame(integration_pred).to_csv("integration_pred_new_method_gex_to_TCR_beta_chain_10X_donor_4_only_022626.csv", index=False)